<a href="https://colab.research.google.com/github/gretadive/correlacion_actividad_solar_elnino/blob/main/04_integracion_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Integración de datos ICEN, ONI y SSN

Este notebook integra las tres series temporales utilizadas en la investigación:

- ICEN: Índice Costero El Niño
- ONI: Oceanic Niño Index
- SSN: Sunspot Number

La integración se realiza mediante la variable temporal `fecha`, conservando únicamente los meses comunes entre las tres series.

In [2]:
from google.colab import userdata
import subprocess
import base64
import os

token = userdata.get("GITHUB_TOKEN")

usuario = "gretadive"
repo = "correlacion_actividad_solar_elnino"
ruta_repo = f"/content/{repo}"

credenciales = base64.b64encode(
    f"{usuario}:{token}".encode()
).decode()

if not os.path.exists(ruta_repo):
    resultado = subprocess.run(
        [
            "git",
            "-c",
            f"http.extraHeader=Authorization: Basic {credenciales}",
            "clone",
            f"https://github.com/{usuario}/{repo}.git"
        ],
        capture_output=True,
        text=True
    )

    if resultado.returncode == 0:
        print("✅ Repositorio conectado correctamente")
    else:
        print("❌ Error al clonar:")
        print(resultado.stderr)
else:
    print("✅ El repositorio ya está disponible")

✅ Repositorio conectado correctamente


In [4]:
%cd /content/correlacion_actividad_solar_elnino
!ls data/processed

/content/correlacion_actividad_solar_elnino
icen_procesado.csv  oni_procesado.csv  README.md  ssn_procesado.csv


In [5]:
import pandas as pd

icen = pd.read_csv(
    "data/processed/icen_procesado.csv",
    parse_dates=["fecha"]
)

oni = pd.read_csv(
    "data/processed/oni_procesado.csv",
    parse_dates=["fecha"]
)

ssn = pd.read_csv(
    "data/processed/ssn_procesado.csv",
    parse_dates=["fecha"]
)

print("ICEN:", icen.shape)
print("ONI :", oni.shape)
print("SSN :", ssn.shape)

ICEN: (917, 4)
ONI : (919, 6)
SSN : (3332, 8)


In [6]:
icen_merge = icen[["fecha", "ICEN"]].copy()
oni_merge = oni[["fecha", "ONI"]].copy()
ssn_merge = ssn[["fecha", "SSN"]].copy()

dataset = pd.merge(
    icen_merge,
    oni_merge,
    on="fecha",
    how="inner",
    validate="one_to_one"
)

dataset = pd.merge(
    dataset,
    ssn_merge,
    on="fecha",
    how="inner",
    validate="one_to_one"
)

dataset = dataset.sort_values("fecha").reset_index(drop=True)

dataset.head()

,fecha,ICEN,ONI,SSN
0,1950-01-01,-0.75,-1.32,143.9
1,1950-02-01,-1.07,-1.20,134.3
2,1950-03-01,-1.25,-1.12,155.4
3,1950-04-01,-1.18,-1.08,160.6
4,1950-05-01,-1.22,-1.10,150.5


In [7]:
print("Total de registros:", len(dataset))
print("Inicio:", dataset["fecha"].min())
print("Fin:", dataset["fecha"].max())

print("\nValores faltantes:")
print(dataset.isnull().sum())

print("\nFechas duplicadas:")
print(dataset["fecha"].duplicated().sum())

Total de registros: 917
Inicio: 1950-01-01 00:00:00
Fin: 2026-05-01 00:00:00

Valores faltantes:
fecha    0
ICEN     0
ONI      0
SSN      0
dtype: int64

Fechas duplicadas:
0


In [8]:
dataset.insert(1, "anio", dataset["fecha"].dt.year)
dataset.insert(2, "mes", dataset["fecha"].dt.month)

dataset.head()

,fecha,anio,mes,ICEN,ONI,SSN
0,1950-01-01,1950,1,-0.75,-1.32,143.9
1,1950-02-01,1950,2,-1.07,-1.20,134.3
2,1950-03-01,1950,3,-1.25,-1.12,155.4
3,1950-04-01,1950,4,-1.18,-1.08,160.6
4,1950-05-01,1950,5,-1.22,-1.10,150.5


In [17]:
dataset.tail()

,fecha,anio,mes,ICEN,ONI,SSN
912,2026-01-01,2026,1,-0.06,-0.39,115.0
913,2026-02-01,2026,2,0.42,-0.21,77.4
914,2026-03-01,2026,3,0.96,0.11,86.6
915,2026-04-01,2026,4,1.34,0.46,79.3
916,2026-05-01,2026,5,1.98,0.95,101.5


In [9]:
fechas_esperadas = pd.date_range(
    start=dataset["fecha"].min(),
    end=dataset["fecha"].max(),
    freq="MS"
)

faltantes = fechas_esperadas.difference(dataset["fecha"])

print("Meses faltantes:", len(faltantes))

if len(faltantes) == 0:
    print("✅ Serie mensual continua")
else:
    print(faltantes)

Meses faltantes: 0
✅ Serie mensual continua


In [10]:
dataset.to_csv(
    "data/processed/dataset_integrado.csv",
    index=False,
    encoding="utf-8"
)

print("✅ Dataset integrado guardado")

✅ Dataset integrado guardado


In [12]:
dataset.describe()

,fecha,anio,mes,ICEN,ONI,SSN
count,917,917.000000,917.000000,917.000000,917.000000,917.000000
mean,1988-03-01 13:14:35.463467840,1987.709924,6.480916,0.002737,0.015213,91.322137
min,1950-01-01 00:00:00,1950.000000,1.000000,-2.230000,-2.040000,0.000000
25%,1969-02-01 00:00:00,1969.000000,3.000000,-0.690000,-0.510000,27.400000
50%,1988-03-01 00:00:00,1988.000000,6.000000,-0.200000,-0.040000,74.800000
75%,2007-04-01 00:00:00,2007.000000,9.000000,0.490000,0.500000,140.800000
max,2026-05-01 00:00:00,2026.000000,12.000000,4.320000,2.590000,359.400000
std,NaN,22.072500,3.455727,1.018758,0.784956,73.625869


In [13]:
!git config user.name "gretadive"
!git config user.email "gretadive@users.noreply.github.com"

!git add data/processed/dataset_integrado.csv
!git commit -m "Integrar datos ICEN ONI y SSN"

[main 5cd0bca] Integrar datos ICEN ONI y SSN
 1 file changed, 918 insertions(+)
 create mode 100644 data/processed/dataset_integrado.csv


In [18]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 917 entries, 0 to 916
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   fecha   917 non-null    datetime64[ns]
 1   anio    917 non-null    int32         
 2   mes     917 non-null    int32         
 3   ICEN    917 non-null    float64       
 4   ONI     917 non-null    float64       
 5   SSN     917 non-null    float64       
dtypes: datetime64[ns](1), float64(3), int32(2)
memory usage: 35.9 KB


In [20]:
print("RESUMEN DEL DATASET INTEGRADO")
print("--------------------------------")
print("Registros:", len(dataset))
print("Periodo:", dataset["fecha"].min(), "a", dataset["fecha"].max())
print("Variables:", list(dataset.columns))
print("Valores faltantes:", dataset.isnull().sum().sum())
print("Fechas duplicadas:", dataset["fecha"].duplicated().sum())

fechas_esperadas = pd.date_range(
    start=dataset["fecha"].min(),
    end=dataset["fecha"].max(),
    freq="MS"
)

faltantes = fechas_esperadas.difference(dataset["fecha"])

print("Meses faltantes:", len(faltantes))

RESUMEN DEL DATASET INTEGRADO
--------------------------------
Registros: 917
Periodo: 1950-01-01 00:00:00 a 2026-05-01 00:00:00
Variables: ['fecha', 'anio', 'mes', 'ICEN', 'ONI', 'SSN']
Valores faltantes: 0
Fechas duplicadas: 0
Meses faltantes: 0


In [23]:
from google.colab import userdata
import subprocess
import base64

token = userdata.get("GITHUB_TOKEN")
usuario = "gretadive"

credenciales = base64.b64encode(
    f"{usuario}:{token}".encode()
).decode()

resultado = subprocess.run(
    [
        "git",
        "-c",
        f"http.extraHeader=Authorization: Basic {credenciales}",
        "push",
        "origin",
        "main"
    ],
    capture_output=True,
    text=True
)

if resultado.returncode == 0:
    print("✅ Dataset integrado enviado correctamente a GitHub")
else:
    print("❌ Error al enviar los cambios:")
    print(resultado.stderr)

✅ Dataset integrado enviado correctamente a GitHub
